
## Unsupervised Learning Models

---

## 1. Supervised vs. Unsupervised Learning

- **Supervised learning:** model learns from labeled data (input → known output).
- **Unsupervised learning:** model learns from **unlabeled data** — there is no target/output variable. The algorithm tries to find hidden structure, patterns, or groupings in the data on its own.

Today's class covered **3 unsupervised learning models**:
1. K-Means Clustering
2. Hierarchical Clustering
3. DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

---

## 2. Clustering — Overview

**Clustering** is an unsupervised approach which finds natural groupings ("clusters") in data such that points within a cluster are more similar to each other than to points in other clusters.

### Approaches to Clustering
- **Agglomerative** — bottom-up approach; starts with each point as its own cluster and merges the closest clusters step by step until one cluster (or the desired number) remains.
- **Divisive** — top-down approach; starts with all points in one cluster and recursively splits them into smaller clusters.

### Agglomerative Clustering — In Detail

Agglomerative is the more commonly used approach (it's what builds the dendrograms seen in the Hierarchical Clustering section below).

**How it works, step by step:**
1. Start by treating **each data point as its own individual cluster** (so N points = N clusters).
2. Find the **two closest clusters** (based on a distance/linkage measure) and **merge them** into a single cluster.
3. Recompute distances between this new cluster and all remaining clusters.
4. **Repeat** steps 2–3, merging the closest pair at each step, until only **one cluster remains** (containing all points) — or until you reach a desired number of clusters.
5. Every merge is recorded, which is what builds the **dendrogram** — the height at which two clusters merge represents the distance between them.

**Linkage criteria** (how "distance between clusters" is defined — determines which clusters get merged):
- **Single linkage:** distance between the *closest* pair of points (one from each cluster). Can produce long, "chained" clusters.
- **Complete linkage:** distance between the *farthest* pair of points (one from each cluster). Tends to produce more compact, evenly-sized clusters.
- **Average linkage:** average distance between all pairs of points across the two clusters.
- **Ward's method:** merges the pair of clusters that leads to the **minimum increase in WCSS** (total within-cluster variance). Very commonly used since it tends to produce balanced clusters.

**Pros:** doesn't require specifying the number of clusters upfront (you decide by cutting the dendrogram); produces a full hierarchy that's easy to visualize and interpret.
**Cons:** computationally expensive for large datasets (distance matrix grows quickly); once two points are merged, that merge can't be undone (greedy — no backtracking).

### Divisive Clustering — In Detail

The reverse of agglomerative — a top-down approach.

**How it works, step by step:**
1. Start with **all data points in a single cluster**.
2. Use an algorithm (e.g., K-Means with K=2) to **split** the cluster into two sub-clusters — typically choosing the split that maximizes the dissimilarity between the resulting groups.
3. **Recursively repeat** the splitting process on each resulting sub-cluster.
4. Continue until each point is its own cluster, or until a stopping criterion is met (e.g., a desired number of clusters, or clusters becoming sufficiently small/homogeneous).

**Pros:** can be more accurate than agglomerative for finding a small number of large, well-separated clusters, since it looks at the "big picture" split first rather than building up from individual points.
**Cons:** computationally more expensive than agglomerative in practice (deciding the *best* split at each step is harder than picking the closest pair to merge) — this is why divisive is used far less often in practice.

**Agglomerative vs. Divisive — Quick Comparison:**

| | Agglomerative | Divisive |
|---|---|---|
| Direction | Bottom-up | Top-down |
| Starts with | N clusters (1 per point) | 1 cluster (all points) |
| Ends with | 1 cluster | N clusters (1 per point) |
| Key operation | Merge closest pair | Split most dissimilar group |
| Common use | Much more widely used | Less common, costlier per step |

---

## 3. K-Means Clustering

K-Means partitions data into **K** clusters, where each point belongs to the cluster with the nearest centroid (mean of the cluster).

### Key Concept: WCSS / Inertia
**WCSS (Within-Cluster Sum of Squares)**, also called **Inertia**, is the **sum of squared distances** between each point and its cluster's centroid. It measures how tightly the points are grouped within a cluster — lower WCSS means tighter, more compact clusters.

### Elbow Method
Used to determine the optimal number of clusters (K).

- The method is based on the relationship between **WCSS (Inertia)** and the **number of clusters (K)**.
- As the number of clusters increases, WCSS steadily decreases (more clusters = points closer to their centroids).
- It's observed that WCSS decreases **steeply at first**, then after a certain number of clusters, the drop **flattens out** and is no longer prominent.
- The point where the curve "bends" (like an elbow) is chosen as the optimal K — adding more clusters beyond this point gives diminishing returns.

*(Diagram in notes: a curve of WCSS vs. K, sharply dropping then flattening — the "elbow" point marks the ideal K.)*

### Other Notes
- **Custom centroid initialization:** how the algorithm's starting centroids are chosen affects the final clustering result (poor initialization can lead to suboptimal clusters — this is why methods like K-Means++ exist to improve initial centroid placement).
- Notebook demo: after running K-Means, an elbow curve was plotted (Inertia vs. K). The curve dropped sharply from K=1→3, then flattened out around **K=4**, marking the elbow point → optimal K = 4 for that dataset.

---

## Cluster Validation Metrics (No External Reference Needed)

These are methods used to measure the **quality of clusters without external labels/references**. There are two aspects to it:

- **Cohesion:** How closely the objects in the *same* cluster are related to each other. It is the **within-cluster sum of squared distances** — the same metric used for K-Means (WCSS):
$$WCSS = \sum \sum (x - m_i)^2$$

- **Separation:** How different the objects in *different* clusters are, and how distinct a well-separated cluster is from other clusters. It is the **between-cluster sum of squared distances (BSS)**:
$$BSS = \sum C_i (m - m_i)^2$$

  where **C** is the size of the individual cluster and **m** is the centroid of all the data points (overall centroid), while **mᵢ** is the centroid of cluster i.

> **Note:** BSS + WSS is always a constant (total variance in the data).

### Silhouette Score

The Silhouette Score measures how well each data point fits within its assigned cluster and how well-separated it is from other clusters. This score is widely used to evaluate clustering algorithms like K-Means. For each point, two key quantities are calculated:

1. **Intra-cluster distance (aᵢ):** the average distance between the data point and all other points in the *same* cluster. A smaller value means the point is closely aligned with its cluster.
2. **Nearest-cluster distance (bᵢ):** the average distance between the data point and all points in the *nearest neighboring* cluster (the next best alternative). A larger value means the point is well-separated from other clusters.

**Silhouette score for a single point:**
$$s(x) = \frac{b(x) - a(x)}{\max\{a(x), b(x)\}}$$

where a(x) is the average distance of x from all other points in the *same* cluster, and b(x) is the average distance of x from all points in the *other* (nearest) cluster.

**Silhouette Coefficient (overall score, averaged across all N points):**
$$SC = \frac{1}{N} \sum S(x)$$

- If **a(x) << b(x)**, the point is much closer to its own cluster than others → indicates **good clustering**.
- Silhouette values range from **-1 to 1**. A higher silhouette score means the **inter-cluster similarity is low** and the **intra-cluster dissimilarity is more** (points are tight within clusters, clusters are far apart) — this is the sign of good clustering.
- Computed in code via: `metrics.silhouette_score(...)` — e.g., a demo run returned a score of **0.972**, indicating very well-separated, cohesive clusters.

---

## 4. Hierarchical Clustering

Builds a hierarchy of clusters, represented visually using a **dendrogram**.

### Dendrogram
- A tree-like diagram that shows how clusters are merged (agglomerative) or split (divisive) at each step.
- To find the optimal number of clusters from a dendrogram, we split it into two parts and look at the resulting distance — this helps determine the **best answer** for cluster count.
- **Where to draw the horizontal line?** — Draw the horizontal cut at the **minimum dendrogram distance possible** (i.e., where the vertical lines being cut are the longest without crossing a merge point) to obtain the best-separated clusters.

### Notebook Demo
- In the demo, `dendrogram(lm)` was plotted using a linkage matrix, producing a tree with two major branches (shown in green and red).
- **"Cutting" the dendrogram** into flat clusters: cutting the tree into **2 clusters** gave a good, clean answer for that dataset.
- This was done in code via `fcluster(lm, 2, criterion='maxclust')`, then plotted with `plotclusters(data, fcluster(...))` to visualize the individual clusters.

---

## 5. DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

An unsupervised machine learning algorithm that defines clusters as **continuous regions of high density**, and marks points in low-density regions as noise (outliers).

### Key Definitions
- **Epsilon (ε / "eps"):** the distance (radius) up to which we look for neighboring points around a given point.
- **Min_points:** the minimum number of points (specified by the user) required within the eps radius for a point to qualify as a "dense" region.
- **Core Point:** if the number of points inside the eps radius of a point is **greater than or equal to** `min_points`, it's called a core point.
- **Border Point:** if the number of points inside the eps radius of a point is **less than** `min_points`, but the point itself lies within the eps radius of a core point, it's called a border point.
- **Noise:** a point that is **neither** a core point **nor** a border point.

**Worked example (from class):** if `eps = 1` and `min_points = 4` — points with ≥4 neighbors within radius 1 become core points (shown in red in the diagram); points that fall in the neighborhood of a core point but don't meet min_points themselves become border points (yellow); isolated points that don't qualify for either become noise (blue).

### Algorithm Steps
1. The algorithm starts with a **random, unvisited point** in the dataset, and its neighboring points are identified based on the eps value.
2. If the point has **≥ min_points** neighbors, cluster formation starts and this point becomes a **core point**; otherwise, it's initially considered **noise**. Note: a point initially classified as noise **can later become a border point** if it falls within the eps radius of a core point discovered later.
3. If the point is a core point, then **all its neighbors become part of the cluster**. If any of those neighboring points are themselves core points, their neighbors are added to the cluster too (this is how the cluster "grows" through connected dense regions).
4. Repeat the steps above until **all points are classified** into different clusters or as noise.

> This algorithm works well when all the clusters are dense enough and are well-separated by low-density regions.

### Applications
- Identifying clusters of arbitrary shape (not just spherical, unlike K-Means).
- Effective at detecting outliers/noise in data.
- Common uses: anomaly detection, geospatial/location data clustering, image segmentation.

---



## Quick Summary Table

| Model | Type | Key Idea | Best For |
|---|---|---|---|
| K-Means | Partition-based | Minimizes WCSS/Inertia via centroids; use Elbow Method to pick K | Spherical, evenly-sized clusters |
| Hierarchical | Tree-based | Builds a dendrogram (agglomerative/divisive) | When you want a hierarchy, no need to pre-specify K |
| DBSCAN | Density-based | Groups dense regions; flags noise/outliers | Arbitrary-shaped clusters, outlier detection |